In [1]:
# ==========================================================
# Notebook 02: Resume Section Extraction
# ==========================================================

import re
import pandas as pd

In [2]:
resumes_df = pd.read_csv("processed/parsed_resumes.csv")

resumes_df.head()

,file_name,raw_text,text_length
0,10554236.pdf,ACCOUNTANT Summary Financial Accountant specia...,24153
1,10674770.pdf,STAFF ACCOUNTANT Summary Highly analytical and...,7488
2,11163645.pdf,ACCOUNTANT Professional Summary To obtain a po...,4736
3,11759079.pdf,SENIOR ACCOUNTANT Experience Company Name June...,5917
4,12065211.pdf,SENIOR ACCOUNTANT Professional Summary Senior ...,5561


In [3]:
sample_resume = resumes_df.loc[0, "raw_text"]

print(sample_resume)

ACCOUNTANT Summary Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense. Highlights Account reconciliations Results-oriented Accounting operations professional Financial reporting Analysis of financial systems Critical thinking ERP (Enterprise Resource Planning) software. Excellent facilitator Accomplishments Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. This allowed for the first successful fiscal year-end close for 2012. In collaboration with DFAS Europe, developed an automated tool that identified duplicate obligations. This tool allowed HQ USAFE to deobligate over $5M in duplicate obligations. Experience Company Name July 2011 to November 2012 Accountant City , State Enterprise Resource Planning Office (ERO) In this position as an Accountant assigned to the Defense Enterprise Accounting and Management System (DEAMS) ERO I was responsible for 

In [4]:
SECTION_PATTERNS = {
    "skills": [
        "skills",
        "technical skills",
        "core competencies",
        "technical expertise",
    ],
    "experience": [
        "experience",
        "work experience",
        "professional experience",
        "employment history",
    ],
    "education": ["education", "academic background", "academic qualifications"],
    "certifications": ["certifications", "certificates", "licenses"],
}

In [5]:
def normalize_text(text):

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    return text

In [6]:
normalized_resume = normalize_text(sample_resume)

print(normalized_resume[:1000])

accountant summary financial accountant specializing in financial planning, reporting and analysis within the department of defense. highlights account reconciliations results-oriented accounting operations professional financial reporting analysis of financial systems critical thinking erp (enterprise resource planning) software. excellent facilitator accomplishments served on a tiger team which identified and resolved general ledger postings in deams totaling $360b in accounting adjustments. this allowed for the first successful fiscal year-end close for 2012. in collaboration with dfas europe, developed an automated tool that identified duplicate obligations. this tool allowed hq usafe to deobligate over $5m in duplicate obligations. experience company name july 2011 to november 2012 accountant city , state enterprise resource planning office (ero) in this position as an accountant assigned to the defense enterprise accounting and management system (deams) ero i was responsible for 

In [7]:
def find_section_position(text, section_keywords):

    positions = []

    for keyword in section_keywords:

        match = re.search(rf"\b{re.escape(keyword)}\b", text)

        if match:

            positions.append(match.start())

    if len(positions) == 0:
        return None

    return min(positions)

In [8]:
def extract_resume_sections(resume_text, section_patterns):

    text = normalize_text(resume_text)

    section_positions = {}

    # Find all section locations
    for section_name, keywords in section_patterns.items():

        position = find_section_position(text, keywords)

        if position is not None:

            section_positions[section_name] = position

    # Sort by appearance
    ordered_sections = sorted(section_positions.items(), key=lambda x: x[1])

    extracted_sections = {}

    for idx, (section_name, start_pos) in enumerate(ordered_sections):

        if idx + 1 < len(ordered_sections):

            end_pos = ordered_sections[idx + 1][1]

        else:

            end_pos = len(text)

        extracted_sections[section_name] = text[start_pos:end_pos].strip()

    return extracted_sections

In [9]:
sections = extract_resume_sections(sample_resume, SECTION_PATTERNS)

sections

{'experience': 'experience company name july 2011 to november 2012 accountant city , state enterprise resource planning office (ero) in this position as an accountant assigned to the defense enterprise accounting and management system (deams) ero i was responsible for identifying and resolving issues affecting the deams general ledger. i worked with teammates from the procure to pay, orders to cash, and budget to report areas to resolve daily challenges encountered with the deployment of deams to additional customers and when system change requests were promoted to production. i supported the testing of scripts, patches, and system change requests ensuring any anomalies were identified to the deams functional management office for action by the deams program management office and/or the system integrator. in addition, i served on a tiger team designed to identify and resolve general ledger posting differences and supported the development of $360b in accounting adjustments allowing for

In [10]:
print("===== SKILLS =====")

print(sections.get("skills", ""))

print("\n===== EXPERIENCE =====")

print(sections.get("experience", ""))

print("\n===== EDUCATION =====")

print(sections.get("education", ""))

===== SKILLS =====
skills accounting; general accounting; accounts payable; program management.

===== EXPERIENCE =====
experience company name july 2011 to november 2012 accountant city , state enterprise resource planning office (ero) in this position as an accountant assigned to the defense enterprise accounting and management system (deams) ero i was responsible for identifying and resolving issues affecting the deams general ledger. i worked with teammates from the procure to pay, orders to cash, and budget to report areas to resolve daily challenges encountered with the deployment of deams to additional customers and when system change requests were promoted to production. i supported the testing of scripts, patches, and system change requests ensuring any anomalies were identified to the deams functional management office for action by the deams program management office and/or the system integrator. in addition, i served on a tiger team designed to identify and resolve general 

In [11]:
parsed_sections = []

for _, row in resumes_df.iterrows():

    extracted = extract_resume_sections(row["raw_text"], SECTION_PATTERNS)

    parsed_sections.append(
        {
            "file_name": row["file_name"],
            "skills": extracted.get("skills", ""),
            "experience": extracted.get("experience", ""),
            "education": extracted.get("education", ""),
            "certifications": extracted.get("certifications", ""),
        }
    )

In [12]:
sections_df = pd.DataFrame(parsed_sections)

sections_df.head()

,file_name,skills,experience,education,certifications
0,10554236.pdf,skills accounting; general accounting; account...,experience company name july 2011 to november ...,education northern maine community college 199...,certifications certified defense financial man...
1,10674770.pdf,skills. highlights dba quick books mas - sage ...,experience staff accountant january 2014 to oc...,"education bachelor of science : accounting , m...",
2,11163645.pdf,skills and attributes. attributes self-motivat...,experience accountant january 2011 to november...,education computer applications specialist cer...,
3,11759079.pdf,skills by drafting over forty memorandums that...,experience company name june 2011 to current s...,"education emory university, goizueta business ...",certifications and awards fulton county casa b...
4,12065211.pdf,skills aderant/cms financial reporting excel u...,experience in full life cycle of general ledge...,education bachelor of business administration ...,"certifications certified public accountant, ne..."


In [13]:
sections_df.to_csv("processed/resume_sections.csv", index=False)

print("Section extraction completed.")

Section extraction completed.


In [14]:
def extract_skill_list(skills_text):

    skills = re.split(r"[,|•|\n]", skills_text)

    skills = [skill.strip() for skill in skills if len(skill.strip()) > 1]

    return skills

In [15]:
sample_skills = extract_skill_list(sections.get("skills", ""))

sample_skills

['skills accounting; general accounting; accounts payable; program management.']